# Hestia A6 — Person 3 final benchmark handoff

Four assigned models; two fixed Brain Dump inputs; four prompting styles. **32 selected generations**, with **40 original generations** preserved including Qwen's failed float16 configuration.

This executed notebook analyzes real Colab outputs; it does not claim that model inference ran again while assembling the report. Scores are explicitly assistant-assisted content judgments, not a blinded or independent human evaluation. The detailed score CSV records each criterion and reason.

**Scope:** completed Person 3 benchmark handoff, ready for Person 1 to integrate. The existing team notebooks are not overwritten.


## Optional model-weight download
The first code cell contains explicit weight-download code. It is disabled during evidence analysis to avoid repeating multi-GB downloads. On a suitably provisioned inference host, set `DOWNLOAD_P3_WEIGHTS=1` before running it. Llama requires approved Hugging Face access and authentication. Never commit model weights or tokens. Downloads alone do not perform benchmarks; use the included runner and documented Colab settings.


In [1]:
from pathlib import Path
import os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/a6-person-3/handoff/manifest.json").exists()), None)
assert ROOT is not None, "Open this notebook from the repository or extracted handoff root"
if os.environ.get("DOWNLOAD_P3_WEIGHTS") == "1":
    from huggingface_hub import snapshot_download
    for model_id in ["microsoft/Phi-3.5-mini-instruct", "meta-llama/Llama-3.2-3B-Instruct",
                     "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"]:
        snapshot_download(model_id, allow_patterns=["*.json", "*.safetensors", "*.model", "*.tiktoken", "*.txt"],
                          ignore_patterns=["original/*"], cache_dir=ROOT / "model_cache/huggingface")
else:
    print("Weight download skipped; analyzing preserved real Colab evidence.")


Weight download skipped; analyzing preserved real Colab evidence.


## Provenance and checks
The original notebook SHA256, T4 GPU, software versions, token cap, precision, prompts and failed outputs are retained. Selected Qwen results use NF4 weights with float32 compute and non-quantized modules; other P3 models use NF4/float16. Configuration changes prevent attributing all observed differences solely to model architecture or size. Timing measures generation, not end-to-end latency.


In [2]:
import csv, json, hashlib, statistics
from collections import Counter
manifest = json.loads((ROOT / "docs/a6-person-3/handoff/manifest.json").read_text())
for relative, expected in manifest["sources"].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
with (ROOT / "a6_results/person3_reviewed_scores.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 32
assert len({(r["model_id"], r["input_id"], r["shot_mode"]) for r in rows}) == 32
assert all(n == 8 for n in Counter(r["model_id"] for r in rows).values())
for row in rows:
    points = [float(row[k]) for k in ["extraction_0_to_3", "type_0_to_2", "priority_0_to_2", "clarity_0_to_2", "format_0_to_1"]]
    assert all(0 <= value <= limit for value, limit in zip(points, [3,2,2,2,1]))
    assert sum(points) == float(row["quality_score_0_to_10"])
print("Source checksums and all 32 score breakdowns verified.")


Source checksums and all 32 score breakdowns verified.


## Scoring policy
Extraction /3, task type /2, priority /2, clarity /2, format /1. Content can earn partial credit even when JSON fails; semantic score is not application acceptance. Invalid JSON must still be rejected by Django. `DEADLINE` and `ERRAND` can both describe a library return. Friday does not establish <=24 hours without today's date. Batteries are not confirmed dead and the book is not confirmed overdue. Tomorrow preparation supports HIGH; URGENT for a leaking tap depends on immediate severity and is treated cautiously.

Format points require full JSON-only output with exact fields; the unchanged shared parser sometimes accepts commentary after a valid object. Hence shared schema passes and format-review points are intentionally distinct. Preserve these limitations when merging with other teammates' scores.


In [3]:
print("Model | Schema passes | Mean reviewed score /10 | Mean generation seconds | Configuration")
for item in manifest["p3_summary"]:
    print(f"{item['model_id']} | {item['schema_passes']}/8 | {item['mean_quality']:.3f} | {item['mean_generation_seconds']:.3f} | {item['configuration']}")
print("\nPrompting comparison (two inputs per mode):")
for model in dict.fromkeys(r["model_id"] for r in rows):
    for mode in ["zero", "one", "few", "many"]:
        group = [r for r in rows if r["model_id"] == model and r["shot_mode"] == mode]
        print(model, mode, "schema", sum(r["schema_valid"] == "True" for r in group), "/2; mean score", statistics.mean(float(r["quality_score_0_to_10"]) for r in group))


Model | Schema passes | Mean reviewed score /10 | Mean generation seconds | Configuration
microsoft/Phi-3.5-mini-instruct | 6/8 | 8.000 | 9.326 | NF4 weights; float16 compute/non-quantized modules
mistralai/Mistral-7B-Instruct-v0.3 | 6/8 | 8.562 | 10.074 | NF4 weights; float16 compute/non-quantized modules
meta-llama/Llama-3.2-3B-Instruct | 1/8 | 7.750 | 6.881 | NF4 weights; float16 compute/non-quantized modules
Qwen/Qwen2.5-7B-Instruct | 8/8 | 9.375 | 12.385 | NF4 weights; float32 compute/non-quantized modules

Prompting comparison (two inputs per mode):
microsoft/Phi-3.5-mini-instruct zero schema 1 /2; mean score 8.0
microsoft/Phi-3.5-mini-instruct one schema 2 /2; mean score 8.0
microsoft/Phi-3.5-mini-instruct few schema 1 /2; mean score 7.75
microsoft/Phi-3.5-mini-instruct many schema 2 /2; mean score 8.25
mistralai/Mistral-7B-Instruct-v0.3 zero schema 1 /2; mean score 8.75
mistralai/Mistral-7B-Instruct-v0.3 one schema 2 /2; mean score 8.5
mistralai/Mistral-7B-Instruct-v0.3 few sch

## Read every scored output
These are original generated strings, including missing braces, unsupported classifications, commentary and truncation. Review explanations are adjacent; no output was repaired.


In [4]:
for row in rows:
    print("\n" + "=" * 72)
    print(row["model_id"], row["input_id"], row["shot_mode"])
    print("Original schema valid:", row["schema_valid"], "Score:", row["quality_score_0_to_10"])
    print("Validation errors:", row["validation_errors"])
    print(row["raw_output"])
    print("Review:", row["review_notes"])



microsoft/Phi-3.5-mini-instruct BD1 zero
Original schema valid: True Score: 9.5
Validation errors: 
{"tasks":[
  {"title":"Kitchen Cleanup","task_type":"CHORE","priority":"MEDIUM","description":"Clean the kitchen before guests arrive."},
  {"title":"Grocery Shopping","task_type":"SHOPPING","priority":"HIGH","description":"Buy necessary groceries for the guests."},
  {"title":"Fix Leaky Tap","task_type":"MAINTENANCE","priority":"HIGH","description":"Repair the bathroom tap to prevent water damage."}
]
}
Review: Assistant-assisted review; E/T/P/C/F=3/2/1.5/2/1. NF4 weights; float16 compute/non-quantized modules. Three explicit tasks, correct types; kitchen MEDIUM understates preparation for tomorrow.

microsoft/Phi-3.5-mini-instruct BD1 one
Original schema valid: True Score: 7.5
Validation errors: 
{"tasks": [
  {"title": "Clean the kitchen", "task_type": "MAINTENANCE", "priority": "MEDIUM", "description": "Tidy up the kitchen before guests arrive."},
  {"title": "Repair the bathroom ta

## Qwen numerical failure and configuration selection
The original eight float16 runs remain archived and are not assigned semantic quality scores. The isolated float16 probe reproduced corruption and measured NaNs during 134 score steps. The same rendered prompt under float32 computation produced strict JSON with no NaN score steps. Both compute dtype and non-quantized module dtype changed; the exact failing layer is not isolated. All eight subsequent float32-compute reruns passed strict JSON/schema checks; they did not record stepwise NaNs. Selecting the usable configuration is explicit, and failed evidence is retained rather than silently discarded.


In [5]:
for suffix in ["float16", "float32"]:
    probe = json.loads((ROOT / f"a6_results/person3_nf4_diagnostics/person3_qwen_probe_{suffix}.json").read_text())
    print(suffix, "NaN score steps:", probe["nan_score_steps"], "schema valid:", probe["schema_valid"], "peak allocated GiB:", round(probe["peak_allocated_gib"], 3))
with (ROOT / "a6_results/person3_nf4_raw/person3_nf4_results.csv").open(newline="") as handle:
    failed_qwen = [r for r in csv.DictReader(handle) if r["model_id"] == "Qwen/Qwen2.5-7B-Instruct"]
for row in failed_qwen:
    print("\nArchived", row["input_id"], row["shot_mode"], "schema", row["schema_valid"])
    print(row["raw_output"])


float16 NaN score steps: 134 schema valid: False peak allocated GiB: 6.586
float32 NaN score steps: 0 schema valid: True peak allocated GiB: 7.707

Archived BD1 zero schema False
{"!token":"v3.1","tasks!":[{"title!":"!!Clean!!!!!Kitchen!","task!!type!":"!!!!!!!!!!!!!!CHORE!","priority!":"!MEDIUM!","description!":"!Clean!!!!the!!!!k!i!t!!h!e!n!e!s!h!t!o!p!prepare!for!!!!!!guest!s!coming!tom!or!w!e!e!k!!"},{"title!":"!!!!Buy!!!!!!!!Groceries!","task!!type!":"!SHOP!PING!","priority!":"!!!!M!I!D!E!!!!!!!!!!!!!!!!!!!!!!!!!!!","description!!":"!!!Need!!basic!gro!c!e!e

Archived BD1 one schema False
{"!token":"c98d7f2b","tasks!":[{"title!":"Clean! the!!!! kitchen!","task!!!_type!":"!!!!!!!!!!!CHORE!","priority!":"!!HIGH!","description!":"!Tidy! up!!!the!!!!k!itchen!for!!!guest!s!coming!!tom!orrow!."},{"title!":"!!!Buy!!!groceries!","!!task!_!type!":"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!SHOP!PING!","!!priority!":"!!!!!!!HIGH!","!description!":"!!Purchase!!!!!!necessary!!items!!for!!the!house!."},{

## Conclusions and team handoff
Qwen NF4/float32 had the strongest structural reliability in these P3 runs, but dentist urgency and deadline judgments still need correction. Mistral was often clear but invented overdue status. Phi misclassified library returns and sometimes added commentary. Llama extracted recognizable tasks quickly but omitted the closing outer brace in seven runs. More examples did not consistently improve quality. Two inputs cannot establish a production-quality winner; keep edit/confirm behavior.

Person 1 should merge `a6_results/person3_benchmark_results.csv` (exact shared 12-column contract, 32 unique cases). Criterion details: `a6_results/person3_reviewed_scores.csv`. Preserve the precision annotations and assistant-assisted review attribution. No fresh inference is needed to execute this report.

**Team notebook integration:** Consolidate P1/P2/P3/P4 into the required final `llm-test/ai_prototype.ipynb`; the current file there is P4-only and root `ai_prototype.ipynb` is P1-only. Retain detailed Part 3 reflections and rerun the final notebook validation. Parameter-size category boundaries are the team's design choice; no particular maximum model size is mandated.


In [6]:
print("P3 handoff verified: 32 selected runs across four assigned models.")
print("40 original generations are preserved, including the eight failed Qwen float16 runs.")
print("Ready for Person 1 to integrate into the team notebook.")

P3 handoff verified: 32 selected runs across four assigned models.
40 original generations are preserved, including the eight failed Qwen float16 runs.
Ready for Person 1 to integrate into the team notebook.
